# Tổng quan và Phân tích Mô tả Dataset

**Đồ án:** Đánh đổi giữa Tăng trưởng Kinh tế và Môi trường: Phân tích đường cong Kuznets và Sự chuyển dịch Năng lượng toàn cầu (2000 - 2022)

Notebook này thực hiện:
1. Giới thiệu tổng quan về dataset
2. Mô tả cấu trúc dữ liệu, số lượng bản ghi và số lượng trường
3. Phân tích thống kê cơ bản

> **Lưu ý:** Bước tiền xử lý dữ liệu (làm sạch, chuyển đổi cấu trúc, v.v.) sẽ được thực hiện **sau** các bước trong notebook này.

---
## 1. Thiết lập môi trường và đọc dữ liệu

In [12]:
import pandas as pd
import numpy as np
import os

# Đường dẫn thư mục Data (chạy từ thư mục gốc của project)
DATA_DIR = 'Data'
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(os.getcwd(), 'Data')
DATASET_PATH = os.path.join(DATA_DIR, 'Dataset.csv')
SERIES_METADATA_PATH = os.path.join(DATA_DIR, 'Series-Metadata.csv')

# Đọc Dataset.csv (nguồn chính)
df = pd.read_csv(DATASET_PATH, encoding='utf-8')

# Đọc Series-Metadata.csv — thử nhiều encoding vì file có thể chứa ký tự đặc biệt (vd: 0x97 = en-dash trong Windows-1252)
encodings = ['utf-8', 'utf-8-sig', 'cp1252', 'latin-1']
df_meta = None
for enc in encodings:
    try:
        df_meta = pd.read_csv(SERIES_METADATA_PATH, encoding=enc, nrows=5000)
        print(f"Đọc Series-Metadata.csv thành công với encoding: {enc}")
        break
    except (UnicodeDecodeError, Exception) as e:
        continue
if df_meta is None:
    print("Không đọc được Series-Metadata.csv với các encoding đã thử:", encodings)

print("Đã load Dataset.csv và Series-Metadata.csv (nếu có).")

Đọc Series-Metadata.csv thành công với encoding: cp1252
Đã load Dataset.csv và Series-Metadata.csv (nếu có).


---
## 2. Giới thiệu tổng quan về Dataset

### 2.1 Nguồn và bối cảnh

- **Nguồn dữ liệu:** World Development Indicators (WDI) – Ngân hàng Thế giới.
- **File sử dụng:**  
  - `Dataset.csv`: bảng dữ liệu chính (quốc gia/khu vực × chỉ số × năm).  
  - `Series-Metadata.csv`: cùng cấu trúc với bảng chính và chứa thêm thông tin mô tả/metadata cho từng series (dùng để tra cứu ý nghĩa chỉ số).
- **Mục đích:** Phân tích mối quan hệ giữa tăng trưởng kinh tế và môi trường (đường cong Kuznets) và chuyển dịch năng lượng toàn cầu trong giai đoạn 2000–2022 (có thể mở rộng đến 2025 tùy mục tiêu).
- **Phạm vi thời gian:** Các cột năm từ **2000** đến **2025** (26 năm).

### 2.2 Các chỉ số (Series) trong dữ liệu

Dataset gồm các chỉ số liên quan đến kinh tế, năng lượng và môi trường, ví dụ:
- **Kinh tế:** GDP bình quân đầu người, tăng trưởng GDP, tỷ trọng công nghiệp.
- **Năng lượng:** Tiếp cận điện, nhiên liệu sạch nấu ăn, sử dụng năng lượng, năng lượng tái tạo.
- **Môi trường:** Phát thải CO2 bình quân đầu người, diện tích rừng.
- **Nhân khẩu:** Dân số (dùng làm trọng số trong một số biểu đồ).

In [14]:
# Liệt kê các Series Name và Series Code có trong dataset
series_info = df[['Series Name', 'Series Code']].drop_duplicates().sort_values('Series Code')
print("Các chỉ số (Series) có trong Dataset.csv:")
print(series_info.to_string(index=False))

Các chỉ số (Series) có trong Dataset.csv:
                                                               Series Name          Series Code
                                              Forest area (% of land area)       AG.LND.FRST.ZS
      Access to clean fuels and technologies for cooking (% of population)       EG.CFT.ACCS.ZS
                                   Access to electricity (% of population)       EG.ELC.ACCS.ZS
        Renewable energy consumption (% of total final energy consumption)       EG.FEC.RNEW.ZS
                              Energy use (kg of oil equivalent per capita)    EG.USE.PCAP.KG.OE
Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita) EN.GHG.CO2.PC.CE.AR5
                 Industry (including construction), value added (% of GDP)       NV.IND.TOTL.ZS
                                                     GDP growth (annual %)    NY.GDP.MKTP.KD.ZG
                                        GDP per capita (constant 2015 US$)       NY.GDP.PCAP.K

---
## 3. Mô tả cấu trúc dữ liệu, số bản ghi và số trường

### 3.1 Cấu trúc bảng

- **Dạng hiện tại:** Mỗi dòng = một tổ hợp **một quốc gia (hoặc khu vực) × một series**.
- **Header:** Dòng đầu tiên chứa tên cột.
- **Cột định danh:** `Country Name`, `Country Code`, `Series Name`, `Series Code`.
- **Cột giá trị theo năm:** `2000 [YR2000]`, `2001 [YR2001]`, …, `2025 [YR2025]` — mỗi ô là giá trị số hoặc chuỗi `".."` (thể hiện dữ liệu thiếu).
- **Cuối file:** Có thể có một số dòng trống hoặc dòng văn bản metadata (ví dụ "Data from database: World Development Indicators") — không được coi là bản ghi dữ liệu.

In [15]:
# Số lượng cột và tên cột
n_cols = len(df.columns)
id_cols = ['Country Name', 'Country Code', 'Series Name', 'Series Code']
year_cols = [c for c in df.columns if c not in id_cols]

print("=== CẤU TRÚC DỮ LIỆU ===")
print(f"Tổng số cột (trường dữ liệu): {n_cols}")
print(f"  - Cột định danh: {len(id_cols)} — {id_cols}")
print(f"  - Cột năm (giá trị): {len(year_cols)} — từ {year_cols[0]} đến {year_cols[-1]}")
print()

# Số dòng: toàn bộ file (kể cả dòng rác)
n_rows_raw = len(df)
print(f"Tổng số dòng trong file (kể cả header đã bỏ): {n_rows_raw}")

# Ước lượng số bản ghi hợp lệ: những dòng có Country Code và Series Code không rỗng và có dạng mã (3 ký tự)
valid_mask = df['Country Code'].notna() & (df['Country Code'].astype(str).str.len() == 3)
df_valid = df[valid_mask]
n_rows_valid = len(df_valid)
print(f"Số bản ghi hợp lệ (có Country Code 3 ký tự): {n_rows_valid}")
print(f"Số dòng có thể là metadata/trống: {n_rows_raw - n_rows_valid}")

=== CẤU TRÚC DỮ LIỆU ===
Tổng số cột (trường dữ liệu): 30
  - Cột định danh: 4 — ['Country Name', 'Country Code', 'Series Name', 'Series Code']
  - Cột năm (giá trị): 26 — từ 2000 [YR2000] đến 2025 [YR2025]

Tổng số dòng trong file (kể cả header đã bỏ): 2665
Số bản ghi hợp lệ (có Country Code 3 ký tự): 2660
Số dòng có thể là metadata/trống: 5


In [16]:
# Hiển thị vài dòng đầu và thông tin kiểu dữ liệu
print("\n--- 5 dòng đầu ---")
display(df.head())

print("\n--- Kiểu dữ liệu (dtypes) ---")
print(df.dtypes)


--- 5 dòng đầu ---


,Country Name,Country Code,Series Name,Series Code,2000 [YR2000],2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],...,2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
0,Afghanistan,AFG,GDP per capita (constant 2015 US$),NY.GDP.PCAP.KD,308.318269746638,277.118051443941,338.139973643387,346.071627096223,338.637273888197,363.640141436773,...,563.872336723147,562.769574140988,553.125151688293,557.861533207459,527.834554499306,408.625855217403,377.665627080705,378.06630312259,..,..
1,Afghanistan,AFG,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,..,-9.4319740700862,28.6000011706788,8.83227780288267,1.41411799339429,11.2297148272859,...,2.26031420279821,2.6470032027451,1.18922812944517,3.91160341625552,-2.35110067203466,-20.7388393676343,-6.24017199240269,2.26694373649188,..,..
2,Afghanistan,AFG,"Industry (including construction), value added...",NV.IND.TOTL.ZS,..,..,23.8101270064854,22.7108641828326,26.2267897500666,26.8120992326398,...,10.4668076938201,10.0518739969063,13.3872469597976,14.0581123697768,12.9525996015547,14.2736570191788,16.0503677223963,13.4498227120978,..,..
3,Afghanistan,AFG,Carbon dioxide (CO2) emissions excluding LULUC...,EN.GHG.CO2.PC.CE.AR5,0.0501382814099344,0.0463511028500998,0.0439140640871224,0.0447454276810823,0.038301143932592,0.052297588398106,...,0.219353479990497,0.2913283907183,0.321418704642259,0.319752781855278,0.310824605884889,0.312156784785117,0.278420956418618,0.280995468771367,0.2829802981146,..
4,Afghanistan,AFG,Forest area (% of land area),AG.LND.FRST.ZS,1.85278199408184,1.85278199408184,1.85278199408184,1.85278199408184,1.85278199408184,1.85278199408184,...,1.85278199408184,1.85278199408184,1.85278199408184,1.85278199408184,1.85278199408184,1.85278199408184,1.85278199408184,1.85278199408184,..,..



--- Kiểu dữ liệu (dtypes) ---
Country Name     str
Country Code     str
Series Name      str
Series Code      str
2000 [YR2000]    str
2001 [YR2001]    str
2002 [YR2002]    str
2003 [YR2003]    str
2004 [YR2004]    str
2005 [YR2005]    str
2006 [YR2006]    str
2007 [YR2007]    str
2008 [YR2008]    str
2009 [YR2009]    str
2010 [YR2010]    str
2011 [YR2011]    str
2012 [YR2012]    str
2013 [YR2013]    str
2014 [YR2014]    str
2015 [YR2015]    str
2016 [YR2016]    str
2017 [YR2017]    str
2018 [YR2018]    str
2019 [YR2019]    str
2020 [YR2020]    str
2021 [YR2021]    str
2022 [YR2022]    str
2023 [YR2023]    str
2024 [YR2024]    str
2025 [YR2025]    str
dtype: object


In [17]:
# Một số dòng cuối (để thấy dòng metadata nếu có)
print("--- 10 dòng cuối (kiểm tra dòng metadata/trống) ---")
display(df.tail(10))

--- 10 dòng cuối (kiểm tra dòng metadata/trống) ---


,Country Name,Country Code,Series Name,Series Code,2000 [YR2000],2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],...,2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
2655,World,WLD,Energy use (kg of oil equivalent per capita),EG.USE.PCAP.KG.OE,1590.53600804969,1584.48166042261,1597.42958388422,1635.68493163054,1682.91262485075,1708.55270571192,...,1784.72552759688,1802.79548497669,1825.73810732344,1828.43822584776,1761.47471149444,1850.99790788266,1853.66774677224,..,..,..
2656,World,WLD,Renewable energy consumption (% of total final...,EG.FEC.RNEW.ZS,17.5262813966932,17.2141012541802,17.2206557859887,17.067470633117,16.7834791518651,16.6370993331808,...,17.6281084915531,17.8541835016377,18.1052270747944,18.5751641385934,19.7355641050053,..,..,..,..,..
2657,World,WLD,Access to electricity (% of population),EG.ELC.ACCS.ZS,78.2214008583172,78.7152432244994,79.1007987332702,79.9649010273987,79.9521476684051,80.7013310816536,...,88.1044791994569,88.9317628469647,89.7973878560124,90.1083920202658,90.3956876080963,91.3342664591051,91.2883104888965,91.5991774421991,..,..
2658,World,WLD,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,48.9258544997835,49.6033207848088,50.2525426502302,50.9502252876777,51.6901468110625,52.4516706578794,...,65.3976911802573,66.8618369839326,68.324904878855,69.6893832205839,71.05987164834,72.2764141456031,73.3585439353749,74.4233687014423,..,..
2659,World,WLD,"Population, total",SP.POP.TOTL,6161884811,6245112906,6327557399,6409750441,6492769815,6575841506,...,7528879985,7614523410,7697233736,7778008621,7854748424,7920514854,7989545217,8064057930,8141808945,..
2660,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2661,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2662,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2663,Data from database: World Development Indicators,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2664,Last Updated: 02/24/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 3.2 Tóm tắt số lượng

| Khái niệm | Giá trị |
|-----------|--------|
| Số cột (trường) | 4 định danh + 26 cột năm = **30** |
| Số bản ghi (dòng dữ liệu hợp lệ) | Theo điều kiện Country Code hợp lệ |
| Số quốc gia/khu vực | Unique `Country Code` |
| Số series | Unique `Series Code` |

In [18]:
n_countries = df_valid['Country Code'].nunique()
n_series = df_valid['Series Code'].nunique()

print("Số quốc gia/khu vực (unique Country Code):", n_countries)
print("Số series (unique Series Code):", n_series)
print("Kỳ vọng số dòng nếu đủ: quốc gia × series =", n_countries * n_series)

Số quốc gia/khu vực (unique Country Code): 266
Số series (unique Series Code): 10
Kỳ vọng số dòng nếu đủ: quốc gia × series = 2660


---
## 4. Phân tích thống kê cơ bản

Phần này thực hiện thống kê mô tả trên dữ liệu **thô** (chưa tiền xử lý): thống kê theo từng cột, giá trị thiếu, và phân bố theo series/quốc gia.

In [19]:
# Chọn các cột năm để thống kê (giá trị có thể là số hoặc "..")
year_columns = [c for c in df_valid.columns if c not in id_cols]

# Chuyển tạm các cột năm sang số (".." -> NaN) để dùng describe()
df_numeric = df_valid.copy()
for col in year_columns:
    df_numeric[col] = pd.to_numeric(df_numeric[col], errors='coerce')

print("=== Thống kê mô tả các cột năm (sau khi chuyển \"..\" -> NaN) ===")
display(df_numeric[year_columns].describe())

=== Thống kê mô tả các cột năm (sau khi chuyển ".." -> NaN) ===


,2000 [YR2000],2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],2006 [YR2006],2007 [YR2007],2008 [YR2008],2009 [YR2009],...,2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
count,2.435000e+03,2.436000e+03,2.446000e+03,2.453000e+03,2.486000e+03,2.488000e+03,2.495000e+03,2.500000e+03,2.475000e+03,2.482000e+03,...,2.496000e+03,2.496000e+03,2.496000e+03,2.494000e+03,2.493000e+03,2.444000e+03,2.295000e+03,2.057000e+03,1.214000e+03,0.0
mean,2.664538e+07,2.701989e+07,2.729195e+07,2.759589e+07,2.760973e+07,2.796866e+07,2.827453e+07,2.860189e+07,2.928642e+07,2.960190e+07,...,3.227960e+07,3.268528e+07,3.308302e+07,3.349995e+07,3.388694e+07,3.490890e+07,3.753960e+07,4.230911e+07,7.244633e+07,NaN
std,2.651024e+08,2.687503e+08,2.718423e+08,2.750675e+08,2.768688e+08,2.803704e+08,2.836057e+08,2.869330e+08,2.920517e+08,2.953612e+08,...,3.212635e+08,3.250330e+08,3.286908e+08,3.323765e+08,3.358080e+08,3.421440e+08,3.559862e+08,3.791173e+08,4.959819e+08,NaN
min,-1.427700e+01,-9.431974e+00,-1.248838e+01,-3.665678e+01,-5.807540e+00,-1.270289e+01,-6.871463e+00,-2.207792e+01,-1.766900e+01,-1.955025e+01,...,-1.704033e+01,-1.567141e+01,-1.965534e+01,-2.765797e+01,-5.440209e+01,-2.073884e+01,-2.875858e+01,-2.943327e+01,-2.655753e+01,NaN
25%,7.700000e+00,7.451361e+00,7.917433e+00,8.556073e+00,9.000000e+00,8.775358e+00,9.185603e+00,9.049045e+00,8.771925e+00,8.307027e+00,...,8.761619e+00,8.972064e+00,8.592239e+00,8.600000e+00,8.506466e+00,9.700000e+00,9.202080e+00,7.706798e+00,3.888326e+00,NaN
50%,4.332021e+01,4.344292e+01,4.392045e+01,4.380000e+01,4.684968e+01,4.715000e+01,4.770000e+01,4.678870e+01,4.551668e+01,4.518943e+01,...,4.558514e+01,4.591357e+01,4.711631e+01,4.719742e+01,4.750000e+01,5.022598e+01,5.860000e+01,5.763669e+01,2.559363e+01,NaN
75%,6.809306e+02,6.713077e+02,7.087427e+02,7.027993e+02,7.484327e+02,7.737207e+02,7.910061e+02,8.148166e+02,8.007449e+02,8.215287e+02,...,8.747835e+02,8.925124e+02,8.927457e+02,9.093926e+02,8.551856e+02,9.998327e+02,1.436815e+03,1.874956e+03,3.293052e+04,NaN
max,6.161885e+09,6.245113e+09,6.327557e+09,6.409750e+09,6.492770e+09,6.575842e+09,6.659977e+09,6.744489e+09,6.830514e+09,6.916589e+09,...,7.528880e+09,7.614523e+09,7.697234e+09,7.778009e+09,7.854748e+09,7.920515e+09,7.989545e+09,8.064058e+09,8.141809e+09,NaN


In [20]:
# Số lượng và tỷ lệ giá trị thiếu theo cột năm
missing_per_year = df_numeric[year_columns].isna().sum()
pct_missing = (missing_per_year / len(df_numeric) * 100).round(2)

missing_df = pd.DataFrame({
    'Năm': [c.split()[0] for c in year_columns],
    'Số giá trị thiếu': missing_per_year.values,
    'Tỷ lệ thiếu (%)': pct_missing.values
})
print("Giá trị thiếu (NaN) theo từng cột năm:")
display(missing_df)

Giá trị thiếu (NaN) theo từng cột năm:


,Năm,Số giá trị thiếu,Tỷ lệ thiếu (%)
0,2000,225,8.46
1,2001,224,8.42
2,2002,214,8.05
3,2003,207,7.78
4,2004,174,6.54
5,2005,172,6.47
6,2006,165,6.20
7,2007,160,6.02
8,2008,185,6.95
9,2009,178,6.69


In [21]:
# Thống kê theo từng Series: count, mean, min, max của các giá trị số (theo cột năm)
stats_by_series = df_numeric.groupby('Series Code')[year_columns].agg(['count', 'mean', 'min', 'max'])
# Đếm theo từng series: số ô có giá trị (không phải NaN)
stats_by_series['valid_count'] = df_numeric.groupby('Series Code')[year_columns].apply(lambda x: x.notna().sum().sum())
stats_by_series['missing_count'] = df_numeric.groupby('Series Code')[year_columns].apply(lambda x: x.isna().sum().sum())
print("Thống kê theo Series (số lượng giá trị hợp lệ, mean, min, max trên các ô năm):")
display(stats_by_series)

Thống kê theo Series (số lượng giá trị hợp lệ, mean, min, max trên các ô năm):


2000 [YR2000]                                           \
                             count          mean          min           max   
Series Code                                                                   
AG.LND.FRST.ZS                 255  3.287414e+01     0.000000  9.557721e+01   
EG.CFT.ACCS.ZS                 237  5.587771e+01     0.000000  1.000000e+02   
EG.ELC.ACCS.ZS                 259  7.445061e+01     2.100000  1.000000e+02   
EG.FEC.RNEW.ZS                 255  3.248294e+01     0.000000  9.790000e+01   
EG.USE.PCAP.KG.OE              192  2.105181e+03   122.876332  1.693872e+04   
EN.GHG.CO2.PC.CE.AR5           251  4.556387e+00     0.000000  1.106476e+02   
NV.IND.TOTL.ZS                 227  2.722410e+01     4.247151  8.479598e+01   
NY.GDP.MKTP.KD.ZG              247  4.552903e+00   -14.277000  5.807810e+01   
NY.GDP.PCAP.KD                 247  1.140725e+04   233.032393  1.204565e+05   
SP.POP.TOTL                    265  2.448235e+08  9544.000000  6.161885e+09   

                     2001 [YR2001]                                           \
                             count          mean          min           max   
Series Code                                                                   
AG.LND.FRST.ZS                 255  3.283484e+01     0.000000  9.555166e+01   
EG.CFT.ACCS.ZS                 237  5.652741e+01     0.100000  1.000000e+02   
EG.ELC.ACCS.ZS                 259  7.500529e+01     1.300000  1.000000e+02   
EG.FEC.RNEW.ZS                 255  3.201253e+01     0.000000  9.830000e+01   
EG.USE.PCAP.KG.OE              192  2.146751e+03   127.088373  1.781191e+04   
EN.GHG.CO2.PC.CE.AR5           251  4.659175e+00     0.000000  1.221372e+02   
NV.IND.TOTL.ZS                 227  2.666481e+01     4.186181  7.741958e+01   
NY.GDP.MKTP.KD.ZG              248  3.243652e+00    -9.431974  6.337988e+01   
NY.GDP.PCAP.KD                 247  1.157327e+04   239.839074  1.231221e+05   
SP.POP.TOTL                    265  2.483665e+08  9586.000000  6.245113e+09   

                     2002 [YR2002]                ... 2024 [YR2024]  \
                             count          mean  ...         count   
Series Code                                       ...                 
AG.LND.FRST.ZS                 255  3.279848e+01  ...             0   
EG.CFT.ACCS.ZS                 237  5.714525e+01  ...             0   
EG.ELC.ACCS.ZS                 260  7.526160e+01  ...             0   
EG.FEC.RNEW.ZS                 257  3.187127e+01  ...             0   
EG.USE.PCAP.KG.OE              192  2.156081e+03  ...             0   
EN.GHG.CO2.PC.CE.AR5           251  4.477349e+00  ...           251   
NV.IND.TOTL.ZS                 230  2.688955e+01  ...           217   
NY.GDP.MKTP.KD.ZG              248  3.454601e+00  ...           241   
NY.GDP.PCAP.KD                 251  1.194884e+04  ...           240   
SP.POP.TOTL                    265  2.518967e+08  ...           265   

                                                              2025 [YR2025]  \
                              mean          min           max         count   
Series Code                                                                   
AG.LND.FRST.ZS                 NaN          NaN           NaN             0   
EG.CFT.ACCS.ZS                 NaN          NaN           NaN             0   
EG.ELC.ACCS.ZS                 NaN          NaN           NaN             0   
EG.FEC.RNEW.ZS                 NaN          NaN           NaN             0   
EG.USE.PCAP.KG.OE              NaN          NaN           NaN             0   
EN.GHG.CO2.PC.CE.AR5  4.401011e+00     0.000000  8.284261e+01             0   
NV.IND.TOTL.ZS        2.639379e+01     2.834462  7.603378e+01             0   
NY.GDP.MKTP.KD.ZG     3.182771e+00   -26.557526  4.381931e+01             0   
NY.GDP.PCAP.KD        1.638524e+04   268.700857  2.471702e+05             0   
SP.POP.TOTL           3.318713e+08  9646.000000  8.141809e+09             0   

             

In [22]:
# Phân bố số bản ghi theo Series Name
print("Số bản ghi theo từng Series:")
record_per_series = df_valid.groupby(['Series Code', 'Series Name']).size().reset_index(name='Số bản ghi')
display(record_per_series)

# Phân bố số bản ghi theo quốc gia (top 15)
print("\nTop 15 quốc gia/khu vực có nhiều bản ghi (mỗi bản ghi = 1 quốc gia × 1 series):")
record_per_country = df_valid.groupby('Country Name').size().sort_values(ascending=False).head(15)
print(record_per_country.to_string())

Số bản ghi theo từng Series:


,Series Code,Series Name,Số bản ghi
0,AG.LND.FRST.ZS,Forest area (% of land area),266
1,EG.CFT.ACCS.ZS,Access to clean fuels and technologies for coo...,266
2,EG.ELC.ACCS.ZS,Access to electricity (% of population),266
3,EG.FEC.RNEW.ZS,Renewable energy consumption (% of total final...,266
4,EG.USE.PCAP.KG.OE,Energy use (kg of oil equivalent per capita),266
5,EN.GHG.CO2.PC.CE.AR5,Carbon dioxide (CO2) emissions excluding LULUC...,266
6,NV.IND.TOTL.ZS,"Industry (including construction), value added...",266
7,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),266
8,NY.GDP.PCAP.KD,GDP per capita (constant 2015 US$),266
9,SP.POP.TOTL,"Population, total",266



Top 15 quốc gia/khu vực có nhiều bản ghi (mỗi bản ghi = 1 quốc gia × 1 series):
Country Name
Afghanistan                    10
Africa Eastern and Southern    10
Africa Western and Central     10
Albania                        10
Algeria                        10
American Samoa                 10
Andorra                        10
Angola                         10
Antigua and Barbuda            10
Arab World                     10
Argentina                      10
Armenia                        10
Aruba                          10
Australia                      10
Austria                        10


In [23]:
# Thống kê mô tả theo từng Series (trung bình giá trị qua các năm)
print("Trung bình giá trị (mean) theo Series và một số năm mẫu (2000, 2010, 2020):")
sample_years = [c for c in year_columns if any(y in c for y in ['2000', '2010', '2020'])]
summary_by_series = df_numeric.groupby('Series Name')[sample_years].mean()
display(summary_by_series.round(4))

Trung bình giá trị (mean) theo Series và một số năm mẫu (2000, 2010, 2020):


,2000 [YR2000],2010 [YR2010],2020 [YR2020]
Series Name,,,
Access to clean fuels and technologies for cooking (% of population),5.587770e+01,6.195240e+01,6.774970e+01
Access to electricity (% of population),7.445060e+01,7.932720e+01,8.629340e+01
Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita),4.556400e+00,5.006000e+00,4.338900e+00
Energy use (kg of oil equivalent per capita),2.105181e+03,2.334532e+03,2.110339e+03
Forest area (% of land area),3.287410e+01,3.258660e+01,3.161160e+01
GDP growth (annual %),4.552900e+00,4.391600e+00,-4.930100e+00
GDP per capita (constant 2015 US$),1.140725e+04,1.426136e+04,1.495002e+04
"Industry (including construction), value added (% of GDP)",2.722410e+01,2.717820e+01,2.457070e+01
"Population, total",2.448235e+08,2.809499e+08,3.187768e+08


### 4.1 Kết luận nhanh

- Dataset có cấu trúc **wide**: mỗi dòng = 1 quốc gia × 1 series, các cột năm chứa giá trị hoặc `".."` (thiếu).
- Số trường: **30** (4 định danh + 26 năm).
- Số bản ghi hợp lệ phụ thuộc vào việc loại bỏ dòng metadata/trống (sẽ xử lý ở bước tiền xử lý).
- Phân tích thống kê cơ bản đã thực hiện: `describe()`, giá trị thiếu theo năm, thống kê theo series và theo quốc gia.

**Bước tiếp theo:** Tiền xử lý dữ liệu (làm sạch, thay `".."` → NaN, lọc series/năm, chuyển sang dạng long/tidy) sẽ được thực hiện trong notebook hoặc bước riêng sau.